Trains the demo/product checkpoint, prioritizing reliable performance on known phrases over open-vocabulary generalization, which the Set A/B results show isn't there yet. This is not the reported adapter. Includes a diagnostic train-set comparison.

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/dysarthria-data'))

In [ ]:
!git clone https://github.com/paloma888/dysarthria-asr-personalization.git

In [ ]:
!pip install transformers datasets peft accelerate evaluate jiwer soundfile librosa torchcodec -q

In [ ]:
import sys
sys.path.append('/content/dysarthria-asr-personalization/training')
from pipeline import processor, dataset_from_pairs, build_training_info, DataCollatorSpeechSeq2Seq

In [ ]:
!cp -r /content/drive/MyDrive/dysarthria-data/MISTY /content/data

In [ ]:
import csv
from pathlib import Path

def read_csv_prompts(folder):
  train_iso, train_cont, val, test_a, test_b = [], [], [], [], []
  with open('/content/dysarthria-asr-personalization/dataset_design/word_prompts.txt', mode='r', encoding='utf-8') as file:
    dict_reader = csv.DictReader(file)
    for row in dict_reader:
      wavpath = Path(folder) / "audio" / row["filename"]
      if not wavpath.exists():
        print(f"doesn't exist: {wavpath}")
      is_isolated = (len(row["text"].split()) == 1)
      wav_prompt_pair = {"audio": str(wavpath), "text": row["text"], "is_isolated": is_isolated}
      if row["split"] == "train":
        if is_isolated:
          train_iso.append(wav_prompt_pair)
        else:
          train_cont.append(wav_prompt_pair)
      else:
        if row["eval_set"].strip() == "A":
          test_a.append(wav_prompt_pair)
        elif row["eval_set"].strip() == "B":
          test_b.append(wav_prompt_pair)

  return train_iso, train_cont, val, test_a, test_b



train_iso, train_cont, val, test_a, test_b = read_csv_prompts("/content/data")
print(f"Train iso: {len(train_iso)}, Train cont: {len(train_cont)}, Set A: {len(test_a)}, Set B: {len(test_b)}")

In [ ]:
import random
random.seed(20)
random.shuffle(train_iso)
random.shuffle(train_cont)

subset_iso = len(train_iso) // 8
subset_cont = len(train_cont) // 8

val = train_iso[:subset_iso] + train_cont[:subset_cont]
train = train_iso[subset_iso:] + train_cont[subset_cont:]

In [ ]:
random.shuffle(train)
random.shuffle(test_a)
random.shuffle(test_b)
random.shuffle(val)

print(f"Train: {len(train)}, Val: {len(val)}, Set A: {len(test_a)}, Set B: {len(test_b)}")

In [ ]:
train_ds = dataset_from_pairs(train)
val_ds = dataset_from_pairs(val)
test_a_ds = dataset_from_pairs(test_a)
test_b_ds = dataset_from_pairs(test_b)


train_ds = train_ds.map(build_training_info)
val_ds = val_ds.map(build_training_info)

import numpy as np
print(np.shape(train_ds[0]["input_features"]))

total_iso_test = sum(1 for p in test_a if p["is_isolated"])
total_cont_test = sum(1 for p in test_a if not p["is_isolated"])

print("iso in set A: ", total_iso_test)
print("cont in set A: ", total_cont_test)

In [ ]:
!pip install -U torchao -q

In [ ]:
!pip install -U hf_transfer -q
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig, PeftModel

base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
base.config.forced_decoder_ids = None
base.config.suppress_tokens = []

base = PeftModel.from_pretrained(base, "/content/drive/MyDrive/clearly-base-adapters/adapter-dropout")
base = base.merge_and_unload()

lora = LoraConfig(r = 8, lora_alpha = 16, target_modules = ["q_proj", "v_proj"], lora_dropout = 0.05)
model = get_peft_model(base, lora)
model.print_trainable_parameters()

In [ ]:
for name, param in model.named_parameters():
  if "encoder.conv1" in name or "encoder.conv2" in name:
    param.requires_grad = True

model.print_trainable_parameters()

In [ ]:
from pipeline import processor
from pipeline import DataCollatorSpeechSeq2Seq
collator = DataCollatorSpeechSeq2Seq(processor)

In [ ]:
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback
training_args = Seq2SeqTrainingArguments(
    output_dir = "/content/drive/MyDrive/checkpoints-personalize-demo",
    per_device_train_batch_size=4,
    learning_rate=1e-4,
    num_train_epochs=15,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    seed=20,
    load_best_model_at_end = False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    warmup_steps = 10,
)

In [ ]:
import transformers
transformers.logging.set_verbosity_error()

In [ ]:
from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model = model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    processing_class= processor.feature_extractor,
)

In [ ]:
trainer.train()

In [ ]:
device = "cuda"
model.to(device)

In [ ]:
from eval import get_metrics, inference
plain_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
plain_model.config.forced_decoder_ids = None
plain_model.config.suppress_tokens = []

plain_train_wer_cont, plain_train_cer_cont, plain_train_wra_iso, plain_train_cer_iso = get_metrics(plain_model, train_ds)

print(f"Train on Plain Whisper Continuous: WER: {plain_train_wer_cont:.2%}, CER: {plain_train_cer_cont:.2%}")
print(f"Train on Plain Whisper Isolated: CER: {plain_train_cer_iso:.2%}, WRA: {plain_train_wra_iso:.2%}")

In [ ]:
clean_base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
clean_base = PeftModel.from_pretrained(clean_base, "/content/drive/MyDrive/clearly-base-adapters/adapter-dropout").merge_and_unload()
clean_base.config.forced_decoder_ids = None
clean_base.config.suppress_tokens = []

base_train_wer_cont, base_train_cer_cont, base_train_wra_iso, base_train_cer_iso = get_metrics(clean_base, train_ds)

print(f"Train on Not-Personalized Continuous: WER: {base_train_wer_cont:.2%}, CER: {base_train_cer_cont:.2%}")
print(f"Train on Not-Personalized Isolated: CER: {base_train_cer_iso:.2%}, WRA: {base_train_wra_iso:.2%}")

In [ ]:
person_train_wer_cont, person_train_cer_cont, person_train_wra_iso, person_train_cer_iso = get_metrics(model, train_ds)

print(f"Train on Personalized Continuous: WER: {person_train_wer_cont:.2%}, CER: {person_train_cer_cont:.2%}")
print(f"Train on Personalized Isolated: CER: {person_train_cer_iso:.2%}, WRA: {person_train_wra_iso:.2%}")

In [ ]:
model.save_pretrained("/content/drive/MyDrive/clearly-base-adapters/adapter-personalize-demo")
import json
with open("/content/drive/MyDrive/log_history_personalize-demo.json","w") as f:
    json.dump(trainer.state.log_history, f, allow_nan=False, default=str)